# Cohort Definition

This notebook defines the clinical cohort for Project Sentinel (Sepsis-Associated AKI Early Warning System).

## Inclusion Criteria
- Patients aged >= 18
- ICU stay >= 24 hours
- Diagnosed with Sepsis (based on clinical criteria)

In [ ]:
import os
import sys
import pandas as pd

# Add the project root to the path so we can import from src
sys.path.append(os.path.abspath('..'))
from src import data_loader

In [ ]:
data_path = '../data/processed/merged_cohort.parquet'
print(f"Loading data from {data_path}...")
try:
    df = pd.read_parquet(data_path)
    print(f"Initial cohort size: {len(df)} patient records")
except FileNotFoundError:
    print(f"File {data_path} not found. Please run the data ingestion notebook first.")
    df = pd.DataFrame() # Fallback for structural purposes

In [ ]:
print("Applying inclusion criteria (e.g., age >= 18, ICU stay >= 24h)...")
if not df.empty:
    df_included = data_loader.apply_inclusion_criteria(df)
    print(f"Cohort size after inclusion criteria: {len(df_included)} records")
else:
    df_included = df

In [ ]:
print("Identifying sepsis onset times...")
if not df_included.empty:
    df_sepsis = data_loader.identify_sepsis_onset(df_included)
    print("Sepsis onset identification complete.")
else:
    df_sepsis = df_included

## Baseline Creatinine Calculation
Baseline creatinine is calculated to determine if a patient has a pre-existing acute kidney injury (AKI) upon admission, which is an exclusion criterion.

The method generally involves:
1. Finding the lowest creatinine value in the 7 days to 365 days prior to admission.
2. If no prior measurements exist, an imputed value based on the MDRD (Modification of Diet in Renal Disease) equation is often used to estimate a baseline.

In [ ]:
print("Defining final cohort by computing baseline creatinine and excluding prevalent AKI...")
if not df_sepsis.empty:
    df_final = data_loader.define_cohort(df_sepsis)
    print(f"Final defined cohort size: {len(df_final)} records")
else:
    df_final = df_sepsis

## Final Cohort Summary

The dataset `df_final` now contains our target cohort for predicting Sepsis-Associated AKI. The cohort has been filtered for adult ICU patients with sepsis, and those with pre-existing end-stage renal disease (ESRD) or prevalent AKI upon admission have been excluded.

The next steps will involve extracting dynamic features (vitals, labs, medications) within the observation window prior to AKI onset.